# Deploy IITB Junta Analytics Genie Space

This notebook deploys the exported Genie Space to your Databricks workspace.

**Prerequisites:**
- Run `python setup.py --catalog YOUR_CATALOG` from the project root to configure catalog references
- Have a SQL warehouse available in your workspace

In [0]:
# Create widgets for parameterization
dbutils.widgets.text("warehouse_id", "d3754c233dd0e522", "SQL Warehouse ID")

In [0]:
# Get widget values
warehouse_id = dbutils.widgets.get("warehouse_id")

# Validate required parameters
if not warehouse_id:
    raise ValueError("warehouse_id is required. Please set it in the widget above.")

print(f"Configuration:")
print(f"  Warehouse ID: {warehouse_id}")

Configuration:
  Warehouse ID: d3754c233dd0e522


In [0]:
import json
from pathlib import Path

# Get the directory where this notebook lives
deploy_dir = "/Workspace/Users/visheshh.arya87@gmail.com/bharatbricksiitb/05-iitb-junta-analytics-genie"

# Read the exported files from workspace
def read_workspace_file(filename: str) -> dict:
    """Read a JSON file from the workspace."""
    file_path = f"{deploy_dir}/{filename}"
    with open(file_path, "r") as f:
        return json.load(f)

# Load exported files
metadata = read_workspace_file("space_metadata.json")

# Load serialized space as string and replace catalog name globally
with open(f"{deploy_dir}/serialized_space.json", "r") as f:
    serialized_space_str = f.read()

# Replace all occurrences of the old catalog with the correct one
old_catalog = "dbdemos_vishesh"
new_catalog = "iitb"
replacement_count = serialized_space_str.count(old_catalog)
serialized_space_str = serialized_space_str.replace(old_catalog, new_catalog)

# Parse back to dict
serialized_space = json.loads(serialized_space_str)

print(f"Loaded Genie Space: {metadata['title']}")
print(f"Description: {metadata['description']}")
print(f"\nReplaced {replacement_count} catalog references from '{old_catalog}' to '{new_catalog}'")
print(f"\nTable references:")
for table in serialized_space.get("data_sources", {}).get("tables", []):
    print(f"  - {table.get('identifier')}")

Loaded Genie Space: 05 IITB Junta Analytics
Description: Yo boss! Explore what the IITB junta is discussing on Reddit. Get fundaes on posting trends, find out which topics cracked it, and discover the studs of the community. Peace types analytics for the r/iitbombay subreddit.

Replaced 16 catalog references from 'dbdemos_vishesh' to 'iitb'

Table references:
  - iitb.bharat_bricks.gold_comments
  - iitb.bharat_bricks.gold_posts


In [0]:
from databricks.sdk import WorkspaceClient

# Initialize Databricks client
w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

# Create the Genie Space using REST API
print(f"\nCreating Genie Space: {metadata['title']}")
print(f"  Warehouse: {warehouse_id}")

# Prepare the request payload
payload = {
    "warehouse_id": warehouse_id,
    "serialized_space": json.dumps(serialized_space),
    "title": metadata["title"],
    "description": metadata["description"]
}

# Make the API call
response = w.api_client.do(
    method="POST",
    path="/api/2.0/genie/spaces",
    body=payload
)

space_id = response["space_id"]

print(f"\n✅ Genie Space created successfully!")
print(f"  Space ID: {space_id}")
print(f"  URL: {w.config.host}/genie/rooms/{space_id}")

Connected to: https://dbc-d3bdf797-f90e.cloud.databricks.com

Creating Genie Space: 05 IITB Junta Analytics
  Warehouse: d3754c233dd0e522

✅ Genie Space created successfully!
  Space ID: 01f12a7143981e3aa6afa6c2f2fe8597
  URL: https://dbc-d3bdf797-f90e.cloud.databricks.com/genie/rooms/01f12a7143981e3aa6afa6c2f2fe8597


In [0]:
# Display clickable link to the Genie Space
genie_url = f"{w.config.host}/genie/rooms/{space_id}"
displayHTML(f'<a href="{genie_url}" target="_blank">Open Genie Space: {metadata["title"]}</a>')

Open Genie Space: 05 IITB Junta Analytics